# Linear regression

The first model, and the one every later model builds on. It predicts a number as a weighted sum of the features. It is simple enough to solve exactly, fast enough to run on millions of rows, and its weights can be read directly. That is why it is still the first thing to try.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from common.datasets import load_california_housing

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

X, y = load_california_housing(n_samples=4000, seed=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

## 1. Intuition

House value rises with income. Plot one against the other and you can draw a straight line through the cloud by eye. Linear regression finds the line that is, on average, closest to the points.

With more features the line becomes a flat sheet (a hyperplane). Each feature gets a weight that says how much the prediction moves when that feature goes up by one unit and the others stay put. A bias (intercept) sets the level when all features are zero.

In [ ]:
plt.scatter(X_train["MedInc"], y_train, s=4, alpha=0.4)
plt.xlabel("MedInc (tens of thousands of dollars)")
plt.ylabel("MedHouseVal ($100k)")
plt.show()

## 2. Maths

For one row, the prediction is a dot product:

$$\hat y = w_0 + w_1 x_1 + \dots + w_d x_d = \mathbf{x}^\top \mathbf{w}$$

Stack the rows into a matrix $X$ (with a column of ones for $w_0$) and the predictions for all rows are $\hat{\mathbf y}=X\mathbf w$.

**Cost function.** The mean squared error is the average squared gap between the line and the points:

$$J(\mathbf w)=\frac1n\lVert \mathbf y-X\mathbf w\rVert^2$$

Squaring punishes big misses far more than small ones and gives a smooth bowl with a single minimum.

**Normal equation.** At the bottom of the bowl the gradient is zero. The gradient is

$$\nabla_{\mathbf w}J=-\frac{2}{n}X^\top(\mathbf y-X\mathbf w)$$

Set it to zero and rearrange:

$$X^\top X\,\mathbf w=X^\top\mathbf y\quad\Rightarrow\quad \hat{\mathbf w}=(X^\top X)^{-1}X^\top\mathbf y$$

This is the exact answer in one step. Its cost grows with the cube of the number of features, and it fails when $X^\top X$ cannot be inverted (for example, two identical columns). Gradient descent is the alternative when there are many features or rows.

**Gradient descent.** Start anywhere and step downhill, repeatedly:

$$\mathbf w\leftarrow\mathbf w-\eta\,\nabla_{\mathbf w}J$$

$\eta$ is the learning rate. This is cost reduction in its simplest form. Notebook 12 digs into the variants.

## 3. From scratch: the normal equation

Add a column of ones, solve the linear system, and compare with sklearn. I use `np.linalg.solve` rather than an explicit inverse. It is faster and more accurate.

In [ ]:
def add_bias(features: np.ndarray) -> np.ndarray:
    return np.column_stack([np.ones(len(features)), features])


def normal_equation(features: np.ndarray, target: np.ndarray) -> np.ndarray:
    A = add_bias(features)
    return np.linalg.solve(A.T @ A, A.T @ target)


w_ne = normal_equation(X_train.to_numpy(), y_train.to_numpy())

lin = LinearRegression().fit(X_train, y_train)
print("normal equation:", w_ne)
print("sklearn        :", np.r_[lin.intercept_, lin.coef_])
print("same answer    :", np.allclose(w_ne, np.r_[lin.intercept_, lin.coef_]))

## 4. From scratch: gradient descent

Same problem, solved by walking downhill. I standardise the features first (the next section shows why). The function keeps the cost at every step so we can plot it.

In [ ]:
def gradient_descent(
    A: np.ndarray, target: np.ndarray, lr: float, steps: int
) -> tuple[np.ndarray, list[float]]:
    n = len(target)
    w = np.zeros(A.shape[1])
    history = []
    for _ in range(steps):
        resid = target - A @ w
        history.append(float(np.mean(resid**2)))
        w = w + lr * (2 / n) * A.T @ resid  # minus the gradient
    return w, history


scaler = StandardScaler().fit(X_train)
A_train = add_bias(scaler.transform(X_train))
w_gd, hist = gradient_descent(A_train, y_train.to_numpy(), lr=0.1, steps=500)

w_exact = normal_equation(scaler.transform(X_train), y_train.to_numpy())
print("max weight gap vs exact:", np.abs(w_gd - w_exact).max())

plt.plot(hist)
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("training MSE")
plt.show()
print("cost at step 0, 10, 499:", hist[0], hist[10], hist[499])

The cost drops fast and then flattens at the minimum. Gradient descent lands on the same weights as the exact solution.

## 5. Feature scaling and why it matters for gradient descent

The gradient of the cost with respect to a weight scales with its feature. A feature measured in thousands (`Population`) produces huge gradients, a feature measured in single digits (`MedInc`) produces small ones. One learning rate cannot suit both. The cost surface is a long, narrow valley, and the steps zigzag across it.

The condition number of $X^\top X$ measures this. A large number means a stretched valley.

In [ ]:
raw = add_bias(X_train.to_numpy())
print("condition number, raw   :", f"{np.linalg.cond(raw.T @ raw):.2e}")
print("condition number, scaled:", f"{np.linalg.cond(A_train.T @ A_train):.2e}")

with np.errstate(all="ignore"):
    w_big, h_big = gradient_descent(raw, y_train.to_numpy(), lr=0.1, steps=50)
print("raw features, lr 0.1, final cost:", h_big[-1])
w_tiny, h_tiny = gradient_descent(raw, y_train.to_numpy(), lr=1e-8, steps=2000)
print("raw features, lr 1e-8, final cost:", h_tiny[-1])
print("scaled, lr 0.1, final cost       :", hist[-1])

On raw features the learning rate that works for scaled data explodes (the cost overflows to `inf`). A rate small enough to stay stable crawls, and after 2,000 steps it is still far from the minimum. Scaling fixes both problems at once.

The normal equation does not care about scale. Gradient descent does, and so do the regularised models in notebook 13.

## 6. The library version

Use sklearn in practice. Put the scaler and the model in a pipeline so the scaler is fitted on training data only. I score on the held-out test set with $R^2$, the share of variance explained. It is 1 for perfect and 0 for "no better than predicting the mean".

In [ ]:
from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
pred = model.predict(X_test)
print("test R^2 :", r2_score(y_test, pred))
print("test MSE :", np.mean((y_test - pred) ** 2))
print("train R^2:", r2_score(y_train, model.predict(X_train)))

## 7. Interpreting the coefficients

Each weight is the change in the prediction for a one-unit rise in that feature, others held fixed. The raw weights are in different units, so they cannot be compared with each other. The weights from scaled features are per one standard deviation, so their sizes are comparable.

In [ ]:
coefs = pd.DataFrame(
    {"raw": lin.coef_, "per_std": model[-1].coef_},
    index=X.columns,
).sort_values("per_std", key=abs, ascending=False)
coefs

`MedInc` has the strongest effect: a one-standard-deviation rise in income adds about 0.8 ($80,000) to the prediction. `Latitude` and `Longitude` are large and negative, which together encode "further north and east is cheaper". They only make sense as a pair, and a straight-line effect of location is a crude summary of the real map. Weights describe the model's association, not cause and effect.

## 8. Tuning

Ordinary linear regression has no hyperparameters to tune. Two things do matter.

- With gradient descent: the learning rate and the number of steps. Watch the cost curve. It should fall smoothly and flatten.
- Which features go in. Notebook 11 covers feature engineering, and notebook 13 adds a penalty that tunes itself against the data.

## 9. Regularisation

Plain least squares has no brake. With many features, few rows or near-duplicate features it chases noise and its weights grow wild (the next sections show it). Regularisation adds a penalty on the size of the weights to the cost. Notebook 13 covers Ridge, Lasso and ElasticNet.

## 10. Assumptions and what breaks

Least squares works well when:

1. The relationship is roughly linear in the features.
2. Errors are independent with roughly constant spread.
3. Errors are roughly bell-shaped (this is what makes squared error the maximum-likelihood cost, see 02_probability/03).
4. Features are not near-duplicates of each other.

Check the residuals (actual minus predicted). A healthy plot is a shapeless cloud around zero.

In [ ]:
resid = y_test - pred
plt.scatter(pred, resid, s=4, alpha=0.4)
plt.axhline(0, c="k", lw=1)
plt.xlabel("predicted")
plt.ylabel("residual")
plt.show()

The diagonal edge is the price cap: values above 5.0 are recorded as 5.0, so large predictions have a hard ceiling on the actual side. The spread also widens with the prediction. Both mean the linear model is imperfect here, and neither is fatal.

### Collinearity

If two features carry almost the same information, the model cannot tell which one deserves the credit. Predictions stay fine but the individual weights become unstable. I add a near-copy of `MedInc` with a little noise, and refit on different bootstrap resamples to see the weights move.

In [ ]:
X_col = X_train.copy()
X_col["MedInc_copy"] = X_col["MedInc"] + rng.normal(0, 0.05, len(X_col))

for name, data in [("original", X_train), ("with near-copy", X_col)]:
    ws = []
    for _ in range(20):
        idx = rng.integers(0, len(data), len(data))
        fit = LinearRegression().fit(data.iloc[idx], y_train.iloc[idx])
        ws.append(fit.coef_[list(data.columns).index("MedInc")])
    print(f"{name:>15}: MedInc weight mean {np.mean(ws):.3f}, sd across resamples {np.std(ws):.3f}")

full = LinearRegression().fit(X_col, y_train)
print("R^2 train, original vs copy:", lin.score(X_train, y_train), full.score(X_col, y_train))

The fit quality barely changes but the `MedInc` weight now swings from one resample to the next, because the credit is split between two columns in an arbitrary way. If you need to interpret weights, drop or combine near-duplicate features. If you only need predictions, collinearity is a smaller problem, and Ridge handles it well.

### Outliers

Squared error punishes a big miss four times as much as a miss half the size, so a few extreme points can drag the whole line. I add 10 wild points to the one-feature fit.

In [ ]:
x1 = X_train[["MedInc"]].to_numpy()[:500]
y1 = y_train.to_numpy()[:500]
clean = LinearRegression().fit(x1, y1)

x_out = np.r_[x1.ravel(), np.full(10, 12.0)].reshape(-1, 1)
y_out = np.r_[y1, np.full(10, 0.2)]
dirty = LinearRegression().fit(x_out, y_out)

print("slope clean:", clean.coef_[0], " slope with 10 outliers:", dirty.coef_[0])
grid = np.linspace(0, 13, 50).reshape(-1, 1)
plt.scatter(x1, y1, s=4, alpha=0.3)
plt.scatter(x_out[-10:], y_out[-10:], c="r", s=15)
plt.plot(grid, clean.predict(grid), label="clean")
plt.plot(grid, dirty.predict(grid), label="with outliers")
plt.legend()
plt.show()

Ten points out of 510 (2%) cut the slope by about 40%. Options: remove points you can show are errors, use a robust loss such as Huber (`HuberRegressor`), or move to absolute error, which section 02 linked to Laplace noise.

## Summary

- Linear regression minimises MSE. The exact answer is the normal equation, and gradient descent reaches it in steps.
- Scale features before gradient descent. On raw housing features the condition number is in the billions and a usable learning rate does not exist.
- Weights are comparable only after scaling, and they describe association, not cause.
- Near-duplicate features make weights unstable. Outliers drag the line.

Next: `11_polynomial_and_features.ipynb`, for when a straight line is not enough.